## Question 5 (Medium) — Order Pipeline with Logging

You are given `orders.csv` with columns `order_id`, `item`, `qty`, `price`.
**Some rows are intentionally invalid** (non-numeric `qty` or `price`, or a
negative value).

Write a function:

```python
def process_orders(csv_path, json_path, log_path):
    ...
```

that:
1. Sets up a logger (`logging.getLogger("orders")`) with a `FileHandler` pointed at `log_path`, level `INFO`.
2. Reads `csv_path` with `csv.DictReader` inside a `try/except/finally`.
3. For each row: convert `qty` and `price` to numbers and compute `total = qty * price`.
   - If conversion fails (`ValueError`) → log an **error** with the row number and reason, and skip the row.
   - If `qty` or `price` is negative → log an **error** and skip the row too.
   - Otherwise → log an **info** message that the row succeeded, and keep the row (including its `total`).
4. Handle `FileNotFoundError` on the input file with a **critical** log message, and return `(0, 0)` in that case.
5. Saves the list of valid, enriched rows (with `total` added) to `json_path` as JSON.
6. Returns a tuple `(num_valid, num_invalid)`.

**Expected result for the sample file below:**
```
(2, 2)
```
(2 valid orders saved to JSON, 2 invalid rows skipped and logged as errors)


In [1]:
# --- SETUP: run this cell first, do not modify ---
import csv

with open("orders.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["order_id", "item", "qty", "price"])
    writer.writerow(["1", "Keyboard", "2", "45.00"])
    writer.writerow(["2", "Mouse", "not_a_number", "15.00"])   # bad qty
    writer.writerow(["3", "Monitor", "1", "-120.00"])           # negative price
    writer.writerow(["4", "USB Cable", "5", "4.50"])

print("orders.csv created (with 2 intentionally bad rows).")


orders.csv created (with 2 intentionally bad rows).


In [31]:
# --- YOUR ANSWER ---
import csv
import json
import logging

def process_orders(csv_path, json_path, log_path):
    # 1. Set up logger + FileHandler(log_path) at level INFO

    logger = logging.getLogger("orders")
    logger.setLevel(logging.INFO)
    logger.handlers.clear()
    
    handler = logging.FileHandler(log_path, "w")
    handler.setFormatter(logging.Formatter("%(asctime)s - %(name)s - %(levelname)s - %(message)s"))
    logger.addHandler(handler)

    num_valid = 0
    num_invalid = 0
    valid_row = []
    
    # 2. try/except/finally around reading csv_path with DictReader
    try:
        with open(csv_path, "r", newline="") as f:
            reader = csv.DictReader(f)
            for row_id, row in enumerate(reader):
                try:
                # 3. Validate qty & price per row; log info/error; collect valid rows with "total"
                    # print(row, row_id)
                    price = float(row["price"])
                    quantity = int(row["qty"])

                except ValueError as e:
                    logger.error(f"Error: {e}")
                    num_invalid += 1
                    continue
                if price < 0 or quantity < 0:
                    logger.error(f"The row with id {row_id} has negative price or quantity")
                    num_invalid += 1
                    continue

                total = price * quantity
                row["total"] = round(total, 2)

                valid_row.append(row)
                num_valid += 1
                logger.info(f"Row {row_id}: Order {row['order_id']} processed successfully")
                

    except FileNotFoundError:
        # 4. On FileNotFoundError: log critical, return (0, 0)
        logger.critical(f"File not found {csv_path}")
        return (0,0)
    # 5. json.dump the valid rows to json_path
    with open(json_path, "w") as w:
        json.dump(valid_row, w, indent=2)
    
    # 6. return (num_valid, num_invalid)
    return (num_valid, num_invalid)


# --- Test your function here ---
print(process_orders("orders.csv", "orders_clean.json", "orders_pipeline.log"))
# expected: (2, 2)


(2, 2)
